# Chapter 7: Bind an answer to a typed reference

**The question.** In this run an LLM answered 70 questions about the results, and a grading function, `answer_score` in `agent_reporting.py`, scored each answer. Before reading those grades in the next notebooks, check that the grader itself works: a wrong number must fail even when it cites the right source, and a malformed answer must not pass.

**How the grader works.** It checks four things separately: whether the answer is right, whether the LLM correctly declined when it had no data (*abstention*), whether the answer has the required format and whether it cites the right sources.

**Result.** The grader gives the intended verdict on all four test answers.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` reran the search with three new optimizer seeds (74101 to 74103), made one live LLM proposal and drew a fresh test sample of 200,000 new customers. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The simulated customers and the frozen model are the same as in the companions, but the seeds, proposal and test sample are new, so do not mix these numbers with the companion's. Rerunning this notebook makes no LLM call.

## Setup and source checks

Finds the saved data for this run and checks, by its SHA-256 fingerprint, that the results file is unchanged.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Test the grader on four made-up answers

An answer must be JSON with two fields: `answer`, a number or null, and `sources`, a list of source identifiers. Anything else is `unscorable`. With sources supplied, a number within $10^{-6}$ of the true value is `correct_answer`. Without sources, or when the result was never measured, the right response is null, graded `correct_abstention`. A separate flag, `provenance_correct`, checks the cited sources. The four test answers are the right value, a wrong value citing the right source, a null answer given no data and a true/false answer.

In [2]:
import sys
sys.path.insert(0,str(LIVE))
from agent_reporting import answer_score
task=dict(answer=.25,sources=['fact:a'])
cases=[({'status':'ok','text':'{"answer":0.25,"sources":["fact:a"]}'},True,'correct_answer'),({'status':'ok','text':'{"answer":0.35,"sources":["fact:a"]}'},True,'wrong_answer'),({'status':'ok','text':'{"answer":null,"sources":[]}'},False,'correct_abstention'),({'status':'ok','text':'{"answer":true,"sources":[]}'},True,'unscorable')]
for rec,access,want in cases:
 score=answer_score(rec,task,access)
 assert score['verdict']==want
 print(want,score)

correct_answer {'verdict': 'correct_answer', 'provenance_correct': True, 'parsed': {'answer': 0.25, 'sources': ['fact:a']}}
wrong_answer {'verdict': 'wrong_answer', 'provenance_correct': True, 'parsed': {'answer': 0.35, 'sources': ['fact:a']}}
correct_abstention {'verdict': 'correct_abstention', 'provenance_correct': True, 'parsed': {'answer': None, 'sources': []}}
unscorable {'verdict': 'unscorable', 'reason': ''}


**Read the output.** **In one line: each test answer gets the intended verdict, so a correct citation or a malformed answer cannot fool the grader.**

**Each printed line** shows the intended verdict, then what the grader returned: `verdict`, `provenance_correct` and the parsed answer.

**Notice.** 0.35 is graded `wrong_answer` even though it cites the correct source, so the right citation cannot rescue a wrong number. The `true` answer is `unscorable` because it is not a number, so it counts neither as correct nor as a decline. These are made-up checks; the grader's live verdicts follow in the next notebooks.